#funadamentos de Datbricks


In [0]:
import pandas as pd


In [0]:
spark.range(1).show()

+---+
| id|
+---+
|  0|
+---+



In [0]:
%sql
CREATE CATALOG IF NOT EXISTS analitica;

In [0]:
%sql
USE CATALOG analitica;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS analitica.data;

In [0]:
%sql
USE CATALOG analitica;

In [0]:
%sql
USE SCHEMA data;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS analitica.data.landing;

In [0]:
%sql
SHOW VOLUMES IN analitica.data;

database,volume_name
data,landing


#2.creacion de tabla sql 


In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple (
    id INT,
    nombre STRING
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple (
    id INT,
    nombre STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO tabla_simple
VALUES
    (1, 'Ana'),
    (2, 'Luis'),
    (3, 'María');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM tabla_simple ORDER BY id;

id,nombre
1,Ana
2,Luis
3,María


#3. Funciones fr pyspark


In [0]:
from pyspark.sql import functions as F
ruta= "/Volumes/analitica/data/landing"

In [0]:
%sql
CREATE OR REPLACE FUNCTION analitica.data.fn_mayor_que10(x DOUBLE)
RETURN X > 10;

In [0]:
%sql
CREATE OR REPLACE FUNCTION analitica.data.fn_mayor_que_10(numero INT)
RETURNS BOOLEAN
RETURN numero > 10;

4. Lectura de datos


In [0]:
path_data_demo= "/Volumes/analitica/data/landing/data_demo.csv"
df = spark.read.csv(
    path_data_demo,
    header=True,
    inferSchema= True
)


In [0]:
display(df.limit(5))

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0


#4.1 guardar tablas 

In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/analitica/data/landing"

df_inmuebles = spark.read.csv(
    ruta + "/data_demo.csv",
    header=True,
    inferSchema=True
)

print("Total de inmuebles:", df_inmuebles.count())

Total de inmuebles: 8969


In [0]:
df_inmuebles.write.format("delta").mode("overwrite").saveAsTable("analitica.data.inmuebles")

In [0]:
%sql
SELECT rooms, garages, useful_area, value
FROM analitica.data.inmuebles
LIMIT 10;

rooms,garages,useful_area,value
3.0,1.0,105.0,1038640.0
3.0,2.0,76.0,606405.0
3.0,1.0,123.0,1534500.0
3.0,2.0,180.0,1131950.0
3.0,1.0,67.0,452672.0
3.0,3.0,198.0,3116290.0
4.0,2.0,245.0,1202560.0
2.0,2.0,66.0,419282.0
2.0,1.0,114.0,1830260.0
2.0,1.0,56.0,274005.0


In [0]:
%sql
SELECT COUNT(*) AS total_inmuebles
FROM analitica.data.inmuebles;

total_inmuebles
8969


#4.2 Lectura de nutrientes y tipos de datos


In [0]:
df_nutrientes = spark.read.csv(
    ruta + "/nutrients_csvfile.csv",
    header=True,
    inferSchema=False
)

display(
    df_nutrientes.select(
        "Food", "Calories", "Fat", "Category"
    ).limit(10)
)

Food,Calories,Fat,Category
Cows' milk,660,40,Dairy products
Milk skim,360,t,Dairy products
Buttermilk,127,5,Dairy products
"Evaporated, undiluted",345,20,Dairy products
Fortified milk,"1,373",42,Dairy products
Powdered milk,515,28,Dairy products
"skim, instant",290,t,Dairy products
"skim, non-instant",290,t,Dairy products
Goats' milk,165,10,Dairy products
(1/2 cup ice cream),690,24,Dairy products


In [0]:
df_nutrientes.printSchema()
print("Alimentos:", df_nutrientes.count())

root
 |-- Food: string (nullable = true)
 |-- Measure: string (nullable = true)
 |-- Grams: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Protein: string (nullable = true)
 |-- Fat: string (nullable = true)
 |-- Sat.Fat: string (nullable = true)
 |-- Fiber: string (nullable = true)
 |-- Carbs: string (nullable = true)
 |-- Category: string (nullable = true)

Alimentos: 335


In [0]:
%sql
SHOW TABLES IN analitica.data;

database,tableName,isTemporary
data,inmuebles,false
data,tabla_simple,false


#4.1 · Lectura de pedidos


In [0]:
df_pedidos = spark.read.csv(
    ruta + "/spark_intro.csv",
    header=True,
    schema="""order_id INT, order_date DATE, country STRING, product STRING,
              category STRING, units_sold INT, unit_price DOUBLE""",
    dateFormat="yyyy-MM-dd",
    mode="FAILFAST"
)

display(df_pedidos)

order_id,order_date,country,product,category,units_sold,unit_price
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0
6,2024-01-08,France,Mouse,Accessories,10,23.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0
9,2024-01-12,France,Monitor,Electronics,1,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0


#4.2 · Selección y filtro

In [0]:
display(
    df_pedidos.select(
        "order_id", "country", "product", "units_sold"
    ).filter(F.col("country") == "Spain")
)

order_id,country,product,units_sold
1,Spain,Laptop,2
2,Spain,Mouse,5
5,Spain,Monitor,2
8,Spain,Keyboard,4


#4.3 · Columna calculada


In [0]:
df_importes = df_pedidos.withColumn(
    "importe",
    F.col("units_sold") * F.col("unit_price")
)

display(df_importes)

order_id,order_date,country,product,category,units_sold,unit_price,importe
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0,2400.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0,125.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0,135.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0,600.0
6,2024-01-08,France,Mouse,Accessories,10,23.0,230.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0,168.0
9,2024-01-12,France,Monitor,Electronics,1,280.0,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0,162.0


#4.4 · Agrupación

In [0]:
resumen_pais = df_importes.groupBy("country").agg(
    F.sum("units_sold").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("importe_total")
)

display(resumen_pais.orderBy(F.desc("importe_total")))

country,unidades,importe_total
Spain,13,3293.0
France,12,1660.0
Germany,10,1597.0


#4.5 · Lectura de la dimensión de productos

In [0]:
df_productos = spark.read.csv(
    ruta + "/dim_spark_intro.csv",
    header=True,
    schema="product STRING, product_family STRING, supplier STRING",
    mode="FAILFAST"
)

display(df_productos)

product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


#4.6 · Unión sencilla y comprobación

In [0]:
display(
    df_productos.groupBy("product")
    .count()
    .filter("count > 1")
)

df_enriquecido = df_importes.join(
    df_productos, "product", "left"
)

display(df_enriquecido)

print("Pedidos antes:", df_pedidos.count())
print("Pedidos después:", df_enriquecido.count())

display(
    df_pedidos.join(
        df_productos, "product", "left_anti"
    )
)

assert df_enriquecido.count() == df_pedidos.count(), \
    "Revisar la llave de productos."

product,count


product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200.0,2400.0,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25.0,125.0,Peripherals,InputMakers
Laptop,3,2024-01-05,France,Electronics,1,1150.0,1150.0,Computers,TechCorp
Keyboard,4,2024-01-06,Germany,Accessories,3,45.0,135.0,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300.0,600.0,Displays,ScreenWorld
Mouse,6,2024-01-08,France,Accessories,10,23.0,230.0,Peripherals,InputMakers
Laptop,7,2024-01-10,Germany,Electronics,1,1300.0,1300.0,Computers,TechCorp
Keyboard,8,2024-01-11,Spain,Accessories,4,42.0,168.0,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280.0,280.0,Displays,ScreenWorld
Mouse,10,2024-01-14,Germany,Accessories,6,27.0,162.0,Peripherals,InputMakers


Pedidos antes: 10
Pedidos después: 10


product,order_id,order_date,country,category,units_sold,unit_price


#4.7 · Guardar y volver a consultar

In [0]:
df_enriquecido.write.format("delta").mode("overwrite").saveAsTable(
    "analitica.data.pedidos_enriquecidos"
)

In [0]:
%sql
SELECT product, product_family, supplier, importe
FROM analitica.data.pedidos_enriquecidos
ORDER BY order_id;

product,product_family,supplier,importe
Laptop,Computers,TechCorp,2400.0
Mouse,Peripherals,InputMakers,125.0
Laptop,Computers,TechCorp,1150.0
Keyboard,Peripherals,InputMakers,135.0
Monitor,Displays,ScreenWorld,600.0
Mouse,Peripherals,InputMakers,230.0
Laptop,Computers,TechCorp,1300.0
Keyboard,Peripherals,InputMakers,168.0
Monitor,Displays,ScreenWorld,280.0
Mouse,Peripherals,InputMakers,162.0


#4.8 · Lectura del ejemplo de ventas

In [0]:
df_ventas = spark.read.csv(
    ruta + "/ventas_2025.csv",
    header=True,
    schema="id INT, fecha DATE, producto STRING, cantidad INT, precio DOUBLE",
    dateFormat="yyyy-MM-dd",
    mode="FAILFAST"
)

display(df_ventas.limit(10))
df_ventas.printSchema()
print("Registros de venta:", df_ventas.count())

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


root
 |-- id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

Registros de venta: 100


#4.9 · Revisión básica antes de guardar

In [0]:
display(
    df_ventas.groupBy("id")
    .count()
    .filter("count > 1")
)

df_invalidos = df_ventas.filter(
    F.col("id").isNull() |
    F.col("fecha").isNull() |
    F.col("producto").isNull() |
    F.col("cantidad").isNull() |
    F.col("precio").isNull() |
    F.isnan("precio") |
    (F.col("cantidad") <= 0) |
    (F.col("precio") < 0)
)

display(df_invalidos)

assert df_invalidos.count() == 0, "Revisar los registros antes de continuar."

id,count


id,fecha,producto,cantidad,precio


#4.10 · Tabla Delta de ventas


In [0]:
df_ventas.write.format("delta").mode("overwrite").saveAsTable(
    "analitica.data.ventas"
)

In [0]:
%sql
DESCRIBE DETAIL analitica.data.ventas;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ffd2b4c0-c67b-4d22-a1c0-b0528391edd8,analitica.data.ventas,null,,2026-10-23T00:45:17.029Z,2026-10-23T00:45:19.000Z,List(),List(),1,2393,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 690a91e4-db92-421b-a364-240709fb9658)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM analitica.data.ventas
GROUP BY producto
ORDER BY importe_total DESC;

producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


Databricks visualization. Run in Databricks to view.

#4.13 · Una variable para cambiar el filtro


In [0]:
producto_seleccionado = "Monitor"

In [0]:
consulta = """
SELECT
    producto,
    SUM(cantidad) AS unidades,
    ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM analitica.data.ventas
WHERE producto = :producto_seleccionado
GROUP BY producto
"""

resultado = spark.sql(
    consulta,
    args={"producto_seleccionado": producto_seleccionado}
)

display(resultado)

producto,unidades,importe_total
Monitor,64,12799.36


In [0]:
%sql
CREATE OR REPLACE TABLE analitica.data.ventas_practica
USING DELTA
AS
SELECT *
FROM analitica.data.ventas;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id = 1;

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99


In [0]:
%sql
UPDATE analitica.data.ventas_practica
SET precio = precio * 1.10
WHERE id = 1;

num_affected_rows
1


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id = 1;

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,28.589000000000002


#INSERT


In [0]:
%sql
INSERT INTO analitica.data.ventas_practica
(id, fecha, producto, cantidad, precio)
VALUES
(1001, DATE '2025-12-15', 'Monitor', 2, 250.00);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id = 1001;

id,fecha,producto,cantidad,precio
1001,2025-12-15,Monitor,2,250.0


#DELETE

In [0]:
%sql
DELETE FROM analitica.data.ventas_practica
WHERE id = 1001;

num_affected_rows
1


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id = 1001;

id,fecha,producto,cantidad,precio


#MERGE


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW nuevos_datos AS
SELECT
    1 AS id,
    DATE '2025-01-03' AS fecha,
    'Teclado' AS producto,
    20 AS cantidad,
    35.00 AS precio

UNION ALL

SELECT
    1002 AS id,
    DATE '2025-12-16' AS fecha,
    'Mouse' AS producto,
    3 AS cantidad,
    45.00 AS precio;

In [0]:
%sql
MERGE INTO analitica.data.ventas_practica AS destino
USING nuevos_datos AS origen
ON destino.id = origen.id

WHEN MATCHED THEN
    UPDATE SET
        destino.fecha = origen.fecha,
        destino.producto = origen.producto,
        destino.cantidad = origen.cantidad,
        destino.precio = origen.precio

WHEN NOT MATCHED THEN
    INSERT (id, fecha, producto, cantidad, precio)
    VALUES (
        origen.id,
        origen.fecha,
        origen.producto,
        origen.cantidad,
        origen.precio
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id IN (1, 1002)
ORDER BY id;

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,35.0
1002,2025-12-16,Mouse,3,45.0


#DESCRIBE HISTORY

In [0]:
%sql
DESCRIBE HISTORY analitica.data.ventas_practica;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-23T01:05:01.000Z,73943875234882,anamria471@gmail.com,MERGE,"Map(predicate -> [""(id#13880 = id#13870)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(586769782286128),3f41b441-7422-4b7f-a22f-b5686ba3ebe4,1023-002352-vmhlhsp-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3267, numTargetBytesRemoved -> 1661, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3500, materializeSourceTimeMs -> 452, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1309, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1654)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
3,2026-10-23T01:03:15.000Z,73943875234882,anamria471@gmail.com,DELETE,"Map(predicate -> [""(id#13618 = 1001)""])",null,List(586769782286128),3a18f72c-70e1-468a-a6ae-455e5bf4d913,1023-002352-vmhlhsp-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1633, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 829, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 683, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 145)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
2,2026-10-23T01:02:08.000Z,73943875234882,anamria471@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(586769782286128),62474805-c5b2-46a3-bf27-90ab329629f8,1023-002352-vmhlhsp-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1633)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1,2026-10-23T01:00:32.000Z,73943875234882,anamria471@gmail.com,UPDATE,"Map(predicate -> [""(id#12836 = 1)""])",null,List(586769782286128),6498d2c8-827d-4ce9-ae2f-70b9e716910b,1023-002352-vmhlhsp-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4387, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2243, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1661, rewriteTimeMs -> 2101)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
0,2026-10-23T00:59:11.000Z,73943875234882,anamria471@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""062a10ae-a98d-4f48-ba99-010b97e27ccc""}, statsOnLoad -> true)",null,List(586769782286128),72e5c831-07db-4137-ab74-8c294be757d7,1023-002352-vmhlhsp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2393)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica VERSION AS OF 0
WHERE id = 1;

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99


In [0]:
%sql
SELECT *
FROM analitica.data.ventas_practica
WHERE id = 1;

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,35.0


In [0]:

%sql
CREATE TABLE IF NOT EXISTS analitica.data.ventas_delta_demo
USING DELTA
AS
SELECT *
FROM analitica.data.ventas;


num_affected_rows,num_inserted_rows


In [0]:

%sql
SELECT COUNT(*) AS total_registros
FROM analitica.data.ventas_delta_demo;


total_registros
100


In [0]:

%sql
INSERT INTO analitica.data.ventas_delta_demo
(id, fecha, producto, cantidad, precio)
VALUES
(1001, DATE '2025-12-15', 'Monitor', 2, 250.00);


num_affected_rows,num_inserted_rows
1,1


In [0]:

%sql
SELECT COUNT(*) AS total_registros
FROM analitica.data.ventas_delta_demo;


total_registros
101


In [0]:

%sql
SELECT 'Pedidos enriquecidos' AS tabla, COUNT(*) AS registros
FROM analitica.data.pedidos_enriquecidos

UNION ALL

SELECT 'Ventas base', COUNT(*)
FROM analitica.data.ventas

UNION ALL

SELECT 'Ensayo Delta', COUNT(*)
FROM analitica.data.ventas_delta_demo;


tabla,registros
Pedidos enriquecidos,10
Ventas base,100
Ensayo Delta,101


#5.1. Creación de vistas SQL

In [0]:

%sql
CREATE OR REPLACE VIEW analitica.data.v_ventas_por_producto AS
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM analitica.data.ventas
GROUP BY producto;


In [0]:

%sql
SELECT *
FROM analitica.data.v_ventas_por_producto
ORDER BY importe_total DESC;


producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


crear

In [0]:

%sql
CREATE OR REPLACE FUNCTION analitica.data.es_venta_alta(
    importe DOUBLE
)
RETURNS BOOLEAN
RETURN importe > 500;


In [0]:

%sql
SELECT
    producto,
    importe_total,
    analitica.data.es_venta_alta(importe_total) AS venta_alta
FROM analitica.data.v_ventas_por_producto
ORDER BY importe_total DESC;


producto,importe_total,venta_alta
Monitor,12799.36,true
Impresora,7641.5,true
Altavoces,6165.0,true
Disco Duro,6159.23,true
Microfono,5579.07,true
Webcam,5478.63,true
Teclado,3378.7,true
Auriculares,2759.08,true
Mouse,1364.0,true
USB,989.01,true


In [0]:

%sql
SELECT id, producto, cantidad, precio,
       analitica.data.es_venta_alta(cantidad * precio) AS supera_umbral
FROM analitica.data.ventas
ORDER BY id
LIMIT 10;


id,producto,cantidad,precio,supera_umbral
1,Teclado,20,25.99,true
2,Webcam,1,39.99,false
3,Auriculares,18,29.99,true
4,Webcam,10,39.99,false
5,Mouse,2,15.5,false
6,Altavoces,13,45.0,true
7,Teclado,11,25.99,false
8,Auriculares,5,29.99,false
9,Monitor,2,199.99,false
10,Microfono,20,59.99,true


In [0]:

%sql
DESCRIBE TABLE EXTENDED analitica.data.v_ventas_por_producto;


col_name,data_type,comment
producto,string,null
unidades,bigint,null
importe_total,double,null
,,
# Detailed Table Information,,
Catalog,analitica,
Database,data,
Table,v_ventas_por_producto,
Owner,anamria471@gmail.com,
Created Time,Fri Oct 23 20:48:59 UTC 2026,


In [0]:

%sql
SHOW GRANTS ON VIEW analitica.data.v_ventas_por_producto;


Principal,ActionType,ObjectType,ObjectKey


In [0]:

%sql
-- GRANT USE CATALOG ON CATALOG analitica TO `lectores_clase`;
-- GRANT USE SCHEMA ON SCHEMA analitica.data TO `lectores_clase`;
-- GRANT SELECT ON VIEW analitica.data.v_ventas_por_producto TO `lectores_clase`;

SHOW GRANTS ON VIEW analitica.data.v_ventas_por_producto;


Principal,ActionType,ObjectType,ObjectKey


In [0]:

%sql
SELECT *
FROM analitica.data.v_ventas_por_producto;


producto,unidades,importe_total
Teclado,130,3378.7
Webcam,137,5478.63
Auriculares,92,2759.08
Mouse,88,1364.0
Altavoces,137,6165.0
Monitor,64,12799.36
Microfono,93,5579.07
USB,99,989.01
Impresora,85,7641.5
Disco Duro,77,6159.23


In [0]:

%sql
SELECT id, fecha, producto, cantidad, precio
FROM analitica.data.ventas_delta_demo
WHERE id IN (1, 1001)
ORDER BY id;


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
1001,2025-12-15,Monitor,2,250.0


![image_1792796915425.png](./image_1792796915425.png "image_1792796915425.png")